 Import des donnees nettoyees (V2) dans PostgreSQL

Chargement etape par etape, avec verification apres chaque table.

1. Import des librairies et connexion

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
from tkinter import Tk
from tkinter.filedialog import askopenfilename
import os

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

print("Connexion configuree.")

Connexion configuree.


2. Selection des fichiers nettoyes (V2)

In [2]:
Tk().withdraw()

print("Choisir clean_customers.csv")
path_customers = askopenfilename(title="Choisir clean_customers.csv")

print("Choisir clean_catalogue.csv")
path_catalogue = askopenfilename(title="Choisir clean_catalogue.csv")

print("Choisir clean_orders.csv")
path_orders = askopenfilename(title="Choisir clean_orders.csv")

print("Choisir clean_transactions.csv")
path_transactions = askopenfilename(title="Choisir clean_transactions.csv")

Choisir clean_customers.csv
Choisir clean_catalogue.csv
Choisir clean_orders.csv
Choisir clean_transactions.csv


3. Lecture des fichiers CSV

In [3]:
df_customers = pd.read_csv(path_customers)
df_catalogue = pd.read_csv(path_catalogue)
df_orders = pd.read_csv(path_orders)
df_transactions = pd.read_csv(path_transactions)

df_customers.shape, df_catalogue.shape, df_orders.shape, df_transactions.shape

((5775, 8), (3757, 6), (9245, 12), (20274, 24))

4. Correction : "Code Client" -> "customer_id_stage"

In [4]:
df_customers = df_customers.rename(columns={"Code Client": "customer_id_stage"})

df_customers.columns.tolist()

['customer_id_stage',
 'customer_type_inferred',
 'wilaya',
 'first_order_date',
 'last_order_date',
 'orders_count',
 'total_amount',
 'average_basket']

 5. Verification et suppression des doublons (customers)

In [5]:
duplicated_customers = df_customers[df_customers.duplicated(subset="customer_id_stage", keep=False)]
print(f"Nombre de lignes avec customer_id_stage duplique : {len(duplicated_customers)}")

df_customers = df_customers.drop_duplicates(subset="customer_id_stage", keep="first")
print(f"Customers apres suppression des doublons : {len(df_customers)} lignes")

Nombre de lignes avec customer_id_stage duplique : 0
Customers apres suppression des doublons : 5775 lignes


 6. Verification et suppression des doublons (catalogue)


In [6]:
duplicated_skus = df_catalogue[df_catalogue.duplicated(subset="sku", keep=False)]
print(f"Nombre de lignes avec SKU duplique : {len(duplicated_skus)}")

df_catalogue = df_catalogue.drop_duplicates(subset="sku", keep="first")
print(f"Catalogue apres suppression des doublons : {len(df_catalogue)} lignes")

Nombre de lignes avec SKU duplique : 6
Catalogue apres suppression des doublons : 3754 lignes


7. Verification et suppression des doublons (orders)

In [7]:
duplicated_orders = df_orders[df_orders.duplicated(subset="order_id_stage", keep=False)]
print(f"Nombre de lignes avec order_id_stage duplique : {len(duplicated_orders)}")

df_orders = df_orders.drop_duplicates(subset="order_id_stage", keep="first")
print(f"Orders apres suppression des doublons : {len(df_orders)} lignes")

Nombre de lignes avec order_id_stage duplique : 0
Orders apres suppression des doublons : 9245 lignes


8. Import customers

In [8]:
df_customers.to_sql("customers", engine, if_exists="append", index=False)
print("Customers importes avec succes.")

Customers importes avec succes.


9. Import catalogue

In [14]:
df_orders.to_sql("orders", engine, if_exists="append", index=False)
print("Orders importees avec succes.")

Orders importees avec succes.


In [ ]:
print("Exemple customer_id dans customers:", df_customers["customer_id_stage"].iloc[0])
print("Exemple customer_id dans orders:", df_orders["customer_id_stage"].iloc[0])

Exemple customer_id dans customers: CLT_S000001
Exemple customer_id dans orders: CLT_S04622


12. Correction du format customer_id_stage (5 chiffres -> 6 chiffres)

Alignement de orders et transactions sur le format customers (6 chiffres).

In [11]:
df_orders["customer_id_stage"] = df_orders["customer_id_stage"].str.replace(
    r"CLT_S(\d{5})$", r"CLT_S0\1", regex=True
)

df_transactions["customer_id_stage"] = df_transactions["customer_id_stage"].str.replace(
    r"CLT_S(\d{5})$", r"CLT_S0\1", regex=True
)

print("Exemple orders apres correction:", df_orders["customer_id_stage"].iloc[0])
print("Exemple transactions apres correction:", df_transactions["customer_id_stage"].iloc[0])

Exemple orders apres correction: CLT_S004622
Exemple transactions apres correction: CLT_S000001


 13. Verification de la correspondance apres correction

In [12]:
customers_ids = set(df_customers["customer_id_stage"].dropna().unique())
orders_ids = set(df_orders["customer_id_stage"].dropna().unique())

matching = orders_ids & customers_ids
missing = orders_ids - customers_ids

print(f"IDs orders qui correspondent a customers : {len(matching)}")
print(f"IDs orders toujours manquants : {len(missing)}")

IDs orders qui correspondent a customers : 4946
IDs orders toujours manquants : 0


15. Import transactions

In [19]:
df_transactions.to_sql("transactions", engine, if_exists="append", index=False)
print("Transactions importees avec succes.")

Transactions importees avec succes.


In [20]:
pd.read_sql("SELECT COUNT(*) FROM customers", engine)

,count
0,5775


In [21]:
pd.read_sql("SELECT COUNT(*) FROM catalogue", engine)


,count
0,3755


In [22]:
pd.read_sql("SELECT COUNT(*) FROM orders", engine)

,count
0,9245


In [23]:
pd.read_sql("SELECT COUNT(*) FROM transactions", engine)

,count
0,20274
